# 04 - Modelo Gold

En este notebook se construye la capa Gold mediante un modelo analítico tipo estrella.

El modelo está compuesto por:

- `fact_anemia`
- `dim_ubicacion`
- `dim_zona`
- `dim_tiempo`

In [0]:
from pyspark.sql.functions import col, row_number, sum as spark_sum, round
from pyspark.sql.window import Window

In [0]:
# Configuración general del proyecto

catalog_name = "anemia_ayacucho"

schema_silver = "silver"
schema_gold = "gold"

spark.sql(f"USE CATALOG {catalog_name}")

DataFrame[]

In [0]:
# Lectura de tablas Silver

df_anemia = spark.table(f"{catalog_name}.{schema_silver}.anemia_limpia")
df_zona = spark.table(f"{catalog_name}.{schema_silver}.dim_zona_limpia")

print("Registros Silver anemia:", df_anemia.count())
print("Registros Silver zona:", df_zona.count())

display(df_anemia.limit(10))

Registros Silver anemia: 1647
Registros Silver zona: 9


ubigeo,provincia,distrito,nro_evaluados,nro_ninos_anemia,anio,zona,fecha_carga,archivo_fuente,tasa_anemia,nivel_riesgo,fecha_proceso_silver,id_registro
1,Cangallo,Cangallo,21,6,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:31:42.386Z,d1b70f172266f7942887034b5ebe55634413b1cfd942d65e3c08a29fa5a1ed73
1,Cangallo,Cangallo,21,6,2023,Zona Noreste,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:31:42.386Z,b7a23b1cfff73fedc874b7a24ce07d70ab02e0208c9cbd63d1dea01ce50c07c6
3,Cangallo,Los Morochucos,41,9,2023,Zona Sur,2026-06-26T06:28:20.042Z,data_2023.csv,21.95,Medio,2026-06-26T06:31:42.386Z,e78d1782cc37d192ad3f745cdc6658323411a3bf32d33fd9e5784eecc72d2578
8,Huamanga,Acos Vinchos,31,8,2023,Zona Suroeste,2026-06-26T06:28:20.042Z,data_2023.csv,25.81,Medio,2026-06-26T06:31:42.386Z,c77d35ed6abe86c0863aeea8a3b977eaa9d59207bcac557304ffb37600780a56
9,Huamanga,Andres Avelino Caceres Dorregaray,123,48,2023,Zona Sureste,2026-06-26T06:28:20.042Z,data_2023.csv,39.02,Medio,2026-06-26T06:31:42.386Z,b7adb6e003a822190d31c2ff425fc63f211adb74eae9e0ea9c804b0cebf45a22
10,Huamanga,Huamanga,366,137,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv,37.43,Medio,2026-06-26T06:31:42.386Z,e5530b2dcc588827044244d0e1853634f470b9671451711d173db23878185126
10,Huamanga,Huamanga,365,137,2023,Zona Noreste,2026-06-26T06:28:20.042Z,data_2023.csv,37.53,Medio,2026-06-26T06:31:42.386Z,c295aaa086d0638434f0cbfa77413216a6db505f758eadd72c0b4505ae82f0b0
11,Huamanga,Carmen Alto,140,54,2023,Zona Este,2026-06-26T06:28:20.042Z,data_2023.csv,38.57,Medio,2026-06-26T06:31:42.386Z,63ee729876a668c872b04c1be5573f344b27fa16d7f2e184062a2c7ef97228d6
15,Huamanga,Pacaycasa,13,3,2023,Zona Este,2026-06-26T06:28:20.042Z,data_2023.csv,23.08,Medio,2026-06-26T06:31:42.386Z,6725f0607d22adf5edbcedf93e766e11294cb6752120e1d792d4ac16c7a022c2
15,Huamanga,Pacaycasa,13,3,2023,Zona Oeste,2026-06-26T06:28:20.042Z,data_2023.csv,23.08,Medio,2026-06-26T06:31:42.386Z,78eef323b71ff26650c5cc219cbc9408c565290a02b70959e26e844a8893ac69


In [0]:
# Creación de dimensión ubicación

w_ubicacion = Window.orderBy("provincia", "distrito", "ubigeo")

df_dim_ubicacion = (
    df_anemia
    .select("ubigeo", "provincia", "distrito")
    .dropDuplicates()
    .withColumn("id_ubicacion", row_number().over(w_ubicacion))
    .select(
        "id_ubicacion",
        col("ubigeo").alias("ubigeo_original"),
        "provincia",
        "distrito"
    )
)

display(df_dim_ubicacion.limit(10))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_ubicacion,ubigeo_original,provincia,distrito
1,1,Cangallo,Cangallo
2,2,Cangallo,Chuschi
3,3,Cangallo,Los Morochucos
4,4,Cangallo,María Parado De Bellido
5,5,Cangallo,Paras
6,6,Cangallo,Totos
7,7,Huamanga,Acocro
8,8,Huamanga,Acos Vinchos
9,9,Huamanga,Andres Avelino Caceres Dorregaray
10,11,Huamanga,Carmen Alto


In [0]:
# Creación de dimensión zona

df_dim_zona = (
    df_zona
    .select(
        "id_zona",
        "zona",
        "abreviatura",
        "tipo_zona",
        "orientacion",
        "descripcion",
        "orden_visualizacion",
        "usar_en_dashboard"
    )
)

display(df_dim_zona)

id_zona,zona,abreviatura,tipo_zona,orientacion,descripcion,orden_visualizacion,usar_en_dashboard
6,Zona Noreste,NE,Intercardinal,Noreste,Clasificación territorial correspondiente al sector noreste del ámbito analizado.,6,true
8,Zona Sureste,SE,Intercardinal,Sureste,Clasificación territorial correspondiente al sector sureste del ámbito analizado.,8,true
9,Zona Suroeste,SO,Intercardinal,Suroeste,Clasificación territorial correspondiente al sector suroeste del ámbito analizado.,9,true
2,Zona Sur,S,Cardinal,Sur,Clasificación territorial correspondiente al sector sur del ámbito analizado.,2,true
3,Zona Este,E,Cardinal,Este,Clasificación territorial correspondiente al sector este del ámbito analizado.,3,true
5,Zona Centro,C,Central,Centro,Clasificación territorial correspondiente al sector central del ámbito analizado.,5,true
4,Zona Oeste,O,Cardinal,Oeste,Clasificación territorial correspondiente al sector oeste del ámbito analizado.,4,true
1,Zona Norte,N,Cardinal,Norte,Clasificación territorial correspondiente al sector norte del ámbito analizado.,1,true
7,Zona Noroeste,NO,Intercardinal,Noroeste,Clasificación territorial correspondiente al sector noroeste del ámbito analizado.,7,true


In [0]:
# Creación de dimensión tiempo

df_dim_tiempo = (
    df_anemia
    .select("anio")
    .dropDuplicates()
    .withColumn("id_tiempo", col("anio"))
    .withColumn("periodo", col("anio").cast("string"))
    .select("id_tiempo", "anio", "periodo")
    .orderBy("anio")
)

display(df_dim_tiempo)

id_tiempo,anio,periodo
2023,2023,2023
2024,2024,2024
2025,2025,2025


In [0]:
# Creación de tabla de hechos fact_anemia

df_fact_anemia = (
    df_anemia
    .join(
        df_dim_ubicacion,
        (df_anemia.ubigeo == df_dim_ubicacion.ubigeo_original) &
        (df_anemia.provincia == df_dim_ubicacion.provincia) &
        (df_anemia.distrito == df_dim_ubicacion.distrito),
        "left"
    )
    .join(df_dim_zona, on="zona", how="left")
    .join(df_dim_tiempo, on="anio", how="left")
    .select(
        df_anemia.id_registro,
        df_dim_ubicacion.id_ubicacion,
        df_dim_zona.id_zona,
        df_dim_tiempo.id_tiempo,
        df_anemia.nro_evaluados,
        df_anemia.nro_ninos_anemia,
        df_anemia.tasa_anemia,
        df_anemia.nivel_riesgo,
        df_anemia.fecha_carga,
        df_anemia.archivo_fuente
    )
)

display(df_fact_anemia.limit(10))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_registro,id_ubicacion,id_zona,id_tiempo,nro_evaluados,nro_ninos_anemia,tasa_anemia,nivel_riesgo,fecha_carga,archivo_fuente
d1b70f172266f7942887034b5ebe55634413b1cfd942d65e3c08a29fa5a1ed73,1,1,2023,21,6,28.57,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
b7a23b1cfff73fedc874b7a24ce07d70ab02e0208c9cbd63d1dea01ce50c07c6,1,6,2023,21,6,28.57,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
e78d1782cc37d192ad3f745cdc6658323411a3bf32d33fd9e5784eecc72d2578,3,2,2023,41,9,21.95,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
c77d35ed6abe86c0863aeea8a3b977eaa9d59207bcac557304ffb37600780a56,8,9,2023,31,8,25.81,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
b7adb6e003a822190d31c2ff425fc63f211adb74eae9e0ea9c804b0cebf45a22,9,8,2023,123,48,39.02,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
e5530b2dcc588827044244d0e1853634f470b9671451711d173db23878185126,12,1,2023,366,137,37.43,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
c295aaa086d0638434f0cbfa77413216a6db505f758eadd72c0b4505ae82f0b0,12,6,2023,365,137,37.53,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
63ee729876a668c872b04c1be5573f344b27fa16d7f2e184062a2c7ef97228d6,10,3,2023,140,54,38.57,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
6725f0607d22adf5edbcedf93e766e11294cb6752120e1d792d4ac16c7a022c2,15,3,2023,13,3,23.08,Medio,2026-06-26T06:28:20.042Z,data_2023.csv
78eef323b71ff26650c5cc219cbc9408c565290a02b70959e26e844a8893ac69,15,4,2023,13,3,23.08,Medio,2026-06-26T06:28:20.042Z,data_2023.csv


In [0]:
# Validación de registros de la tabla principal

print("Registros de fact_anemia:", df_fact_anemia.count())

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registros de fact_anemia: 1647


In [0]:
# Escritura de dimensiones y tabla de hechos en Gold

(
    df_dim_ubicacion.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_ubicacion")
)

(
    df_dim_zona.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_zona")
)

(
    df_dim_tiempo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.dim_tiempo")
)

(
    df_fact_anemia.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.fact_anemia")
)

print("Tablas Gold creadas correctamente.")

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tablas Gold creadas correctamente.


In [0]:
# Construcción de tabla agregada para dashboard

df_resumen_dashboard = (
    df_fact_anemia
    .join(df_dim_tiempo, on="id_tiempo", how="left")
    .join(df_dim_zona, on="id_zona", how="left")
    .join(df_dim_ubicacion, on="id_ubicacion", how="left")
    .groupBy("anio", "provincia", "distrito", "zona", "nivel_riesgo")
    .agg(
        spark_sum("nro_evaluados").alias("total_evaluados"),
        spark_sum("nro_ninos_anemia").alias("total_ninos_anemia")
    )
    .withColumn(
        "tasa_anemia",
        round((col("total_ninos_anemia") / col("total_evaluados")) * 100, 2)
    )
)

(
    df_resumen_dashboard.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_gold}.resumen_dashboard")
)

display(df_resumen_dashboard.limit(10))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


anio,provincia,distrito,zona,nivel_riesgo,total_evaluados,total_ninos_anemia,tasa_anemia
2023,Huamanga,Huamanga,Zona Noreste,Medio,365,137,37.53
2023,Huamanga,Pacaycasa,Zona Oeste,Medio,13,3,23.08
2023,Huamanga,San Jose De Ticllas,Zona Sureste,Bajo,5,0,0.0
2023,Huamanga,Ocros,Zona Noreste,Medio,27,10,37.04
2023,Huamanga,San Jose De Ticllas,Zona Centro,Bajo,5,0,0.0
2023,Huanta,Llochegua,Zona Oeste,Bajo,54,8,14.81
2023,Lucanas,Aucara,Zona Norte,Alto,6,4,66.67
2023,Huamanga,Santiago De Pischa,Zona Norte,Alto,5,2,40.0
2023,Huamanga,Socos,Zona Este,Alto,29,21,72.41
2023,Huanta,Luricocha,Zona Sur,Medio,39,14,35.9


In [0]:
# Creación de vistas SQL para análisis y dashboard

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{schema_gold}.vw_kpi_general AS
SELECT
    SUM(f.nro_evaluados) AS total_evaluados,
    SUM(f.nro_ninos_anemia) AS total_ninos_anemia,
    ROUND((SUM(f.nro_ninos_anemia) / SUM(f.nro_evaluados)) * 100, 2) AS tasa_anemia_general,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT u.provincia) AS total_provincias,
    COUNT(DISTINCT u.distrito) AS total_distritos
FROM {catalog_name}.{schema_gold}.fact_anemia f
LEFT JOIN {catalog_name}.{schema_gold}.dim_ubicacion u
ON f.id_ubicacion = u.id_ubicacion
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{schema_gold}.vw_evolucion_anual AS
SELECT
    t.anio,
    SUM(f.nro_evaluados) AS total_evaluados,
    SUM(f.nro_ninos_anemia) AS total_ninos_anemia,
    ROUND((SUM(f.nro_ninos_anemia) / SUM(f.nro_evaluados)) * 100, 2) AS tasa_anemia
FROM {catalog_name}.{schema_gold}.fact_anemia f
LEFT JOIN {catalog_name}.{schema_gold}.dim_tiempo t
ON f.id_tiempo = t.id_tiempo
GROUP BY t.anio
ORDER BY t.anio
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{schema_gold}.vw_anemia_provincia AS
SELECT
    u.provincia,
    SUM(f.nro_evaluados) AS total_evaluados,
    SUM(f.nro_ninos_anemia) AS total_ninos_anemia,
    ROUND((SUM(f.nro_ninos_anemia) / SUM(f.nro_evaluados)) * 100, 2) AS tasa_anemia
FROM {catalog_name}.{schema_gold}.fact_anemia f
LEFT JOIN {catalog_name}.{schema_gold}.dim_ubicacion u
ON f.id_ubicacion = u.id_ubicacion
GROUP BY u.provincia
ORDER BY total_ninos_anemia DESC
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{schema_gold}.vw_anemia_zona AS
SELECT
    z.zona,
    z.tipo_zona,
    SUM(f.nro_evaluados) AS total_evaluados,
    SUM(f.nro_ninos_anemia) AS total_ninos_anemia,
    ROUND((SUM(f.nro_ninos_anemia) / SUM(f.nro_evaluados)) * 100, 2) AS tasa_anemia
FROM {catalog_name}.{schema_gold}.fact_anemia f
LEFT JOIN {catalog_name}.{schema_gold}.dim_zona z
ON f.id_zona = z.id_zona
GROUP BY z.zona, z.tipo_zona, z.orden_visualizacion
ORDER BY z.orden_visualizacion
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{schema_gold}.vw_ranking_distritos_criticos AS
SELECT
    u.provincia,
    u.distrito,
    SUM(f.nro_evaluados) AS total_evaluados,
    SUM(f.nro_ninos_anemia) AS total_ninos_anemia,
    ROUND((SUM(f.nro_ninos_anemia) / SUM(f.nro_evaluados)) * 100, 2) AS tasa_anemia
FROM {catalog_name}.{schema_gold}.fact_anemia f
LEFT JOIN {catalog_name}.{schema_gold}.dim_ubicacion u
ON f.id_ubicacion = u.id_ubicacion
GROUP BY u.provincia, u.distrito
ORDER BY tasa_anemia DESC, total_ninos_anemia DESC
""")

print("Vistas SQL creadas correctamente.")

Vistas SQL creadas correctamente.


In [0]:
# Validación de tablas y vistas Gold

display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_gold}"))

database,tableName,isTemporary
gold,dim_tiempo,false
gold,dim_ubicacion,false
gold,dim_zona,false
gold,fact_anemia,false
gold,resumen_dashboard,false
gold,vw_anemia_provincia,false
gold,vw_anemia_zona,false
gold,vw_evolucion_anual,false
gold,vw_kpi_general,false
gold,vw_ranking_distritos_criticos,false


## Evidencia esperada

En este notebook se debe capturar:

- Modelo estrella construido.
- Tabla principal `fact_anemia`.
- Dimensiones `dim_ubicacion`, `dim_zona` y `dim_tiempo`.
- Tabla agregada para dashboard.
- Vistas SQL creadas en Gold.